# 08 - Baseline Models for NO2

This notebook computes three baseline predictors on the final 3-month test period (2024-07-01 to 2024-09-30):

1. Climatological monthly mean
2. Cosine seasonal model based on day-of-year (DOY)
3. Persistence model (`NO2` at `t-1`)

It reports MAE, RMSE, and R2 for each baseline.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.load_airnow import load_all, FULL_TRAIN_END

TEST_START = pd.Timestamp("2024-07-01 00:00")
TEST_END = pd.Timestamp("2024-09-30 23:00")

df = load_all().clip(lower=0.0)
df = df.sort_index()

train_df = df[df.index <= FULL_TRAIN_END]
test_df = df[(df.index >= TEST_START) & (df.index <= TEST_END)]

print(f"Train range: {train_df.index.min()} -> {train_df.index.max()} ({len(train_df):,} hours)")
print(f"Test range : {test_df.index.min()} -> {test_df.index.max()} ({len(test_df):,} hours)")
print(f"Sites      : {test_df.shape[1]}")

Train range: 2023-07-01 00:00:00 -> 2024-06-30 23:00:00 (8,784 hours)
Test range : 2024-07-01 00:00:00 -> 2024-09-30 23:00:00 (2,208 hours)
Sites      : 197


In [2]:
def metrics_from_frames(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame) -> dict:
    y_true = y_true_df.values.astype(float)
    y_pred = y_pred_df.values.astype(float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = y_true[mask]
    yp = y_pred[mask]

    mae = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))

    ss_res = float(np.sum((yt - yp) ** 2))
    y_bar = float(np.mean(yt))
    ss_tot = float(np.sum((yt - y_bar) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {"MAE": mae, "RMSE": rmse, "R2": r2}

# 1) Climatological monthly mean baseline
monthly_means = train_df.groupby(train_df.index.month).mean()
global_site_mean = train_df.mean()

monthly_rows = []
for ts in test_df.index:
    m = ts.month
    if m in monthly_means.index:
        row = monthly_means.loc[m]
    else:
        row = global_site_mean
    monthly_rows.append(row.values)

pred_monthly = pd.DataFrame(monthly_rows, index=test_df.index, columns=test_df.columns)

# 2) Cosine DOY seasonal baseline
def doy_fraction(idx: pd.DatetimeIndex) -> np.ndarray:
    return idx.dayofyear.values + idx.hour.values / 24.0

doy_train = doy_fraction(train_df.index)
doy_test = doy_fraction(test_df.index)

omega = 2.0 * np.pi / 365.25
X_train = np.column_stack([
    np.ones_like(doy_train),
    np.cos(omega * doy_train),
    np.sin(omega * doy_train),
])
X_test = np.column_stack([
    np.ones_like(doy_test),
    np.cos(omega * doy_test),
    np.sin(omega * doy_test),
])

cos_preds = np.full((len(test_df), test_df.shape[1]), np.nan, dtype=float)

for j, col in enumerate(train_df.columns):
    y = train_df[col].values.astype(float)
    valid = np.isfinite(y)

    if valid.sum() < 10:
        # Not enough observations; fallback to site mean.
        site_mean = np.nanmean(y) if np.isfinite(np.nanmean(y)) else 0.0
        cos_preds[:, j] = site_mean
        continue

    beta, _, _, _ = np.linalg.lstsq(X_train[valid], y[valid], rcond=None)
    cos_preds[:, j] = X_test @ beta

pred_cosine = pd.DataFrame(cos_preds, index=test_df.index, columns=test_df.columns)
pred_cosine = pred_cosine.clip(lower=0.0)

# 3) Persistence baseline: NO2(t-1 hour)
pred_persistence = df.shift(1).loc[test_df.index]

scores = {
    "Monthly Climatology": metrics_from_frames(test_df, pred_monthly),
    "DOY Cosine": metrics_from_frames(test_df, pred_cosine),
    "Persistence (t-1)": metrics_from_frames(test_df, pred_persistence),
}

results = pd.DataFrame(scores).T
results = results[["MAE", "RMSE", "R2"]]
results_rounded = results.copy()
results_rounded["MAE"] = results_rounded["MAE"].round(4)
results_rounded["RMSE"] = results_rounded["RMSE"].round(4)
results_rounded["R2"] = results_rounded["R2"].round(4)

results_rounded

/tmp/ipykernel_3764793/3095114056.py:61: RuntimeWarning: Mean of empty slice
  site_mean = np.nanmean(y) if np.isfinite(np.nanmean(y)) else 0.0


,MAE,RMSE,R2
Monthly Climatology,3.3832,5.1908,0.4595
DOY Cosine,3.5408,5.3257,0.4149
Persistence (t-1),1.5846,2.8348,0.8337


In [3]:
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "baseline_models_comparison.csv"
results.to_csv(out_path, index=True)
print(f"Saved baseline comparison -> {out_path}")

Saved baseline comparison -> /mnt/data3/isybelle1118/NO2 Forecasting/outputs/baseline_models_comparison.csv


## Meteorological Feature Analysis for NO2

This section analyzes meteorological predictors against NO2 using EPA AQS daily files:

- Temperature
- Wind Speed
- Surface Pressure
- Relative Humidity

Outputs:

1. Pearson correlation matrix
2. Pairwise scatter plots of NO2 vs each meteorological feature

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")

analysis_dir = ROOT / "no2 analysis"

no2_files = [analysis_dir / "daily_42602_2023.csv", analysis_dir / "daily_42602_2024.csv"]
met_files = [
    analysis_dir / "daily_WIND_2023.csv", analysis_dir / "daily_WIND_2024.csv",
    analysis_dir / "daily_TEMP_2023.csv", analysis_dir / "daily_TEMP_2024.csv",
    analysis_dir / "daily_PRESS_2023.csv", analysis_dir / "daily_PRESS_2024.csv",
    analysis_dir / "daily_RH_DP_2023.csv", analysis_dir / "daily_RH_DP_2024.csv",
]

missing = [str(p) for p in (no2_files + met_files) if not p.exists()]

if missing:
    print("Meteorological source CSV files were not found in 'no2 analysis'.")
    print("Skipping met-feature analysis and regression sections until these files are added:")
    for m in missing:
        print(f"  - {m}")
    analysis_df = pd.DataFrame()
else:
    def _load_concat(paths):
        frames = []
        for p in paths:
            df_ = pd.read_csv(p, dtype={"State Code": str, "County Code": str, "Site Num": str}, low_memory=False)
            frames.append(df_)
        return pd.concat(frames, ignore_index=True)

    no2_raw = _load_concat(no2_files)
    met_raw = _load_concat(met_files)

    for d in (no2_raw, met_raw):
        d["Date Local"] = pd.to_datetime(d["Date Local"], errors="coerce")
        d["site_id"] = (
            d["State Code"].str.zfill(2)
            + "-"
            + d["County Code"].str.zfill(3)
            + "-"
            + d["Site Num"].str.zfill(4)
        )

    # NO2 daily mean by site and day
    no2_daily = (
        no2_raw.groupby(["site_id", "Date Local"], as_index=False)["Arithmetic Mean"]
        .mean()
        .rename(columns={"Arithmetic Mean": "NO2"})
    )

    param = met_raw["Parameter Name"].fillna("").str.lower()
    met_raw = met_raw.copy()
    met_raw["feature"] = np.select(
        [
            param.str.contains("wind speed", na=False),
            param.str.contains("outdoor temperature|temperature", na=False),
            param.str.contains("barometric pressure", na=False),
            param.str.contains("relative humidity", na=False),
        ],
        ["Wind Speed", "Temperature", "Surface Pressure", "Relative Humidity"],
        default="Other",
    )

    met_use = met_raw[met_raw["feature"].isin(["Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity"])].copy()

    met_daily = (
        met_use.groupby(["site_id", "Date Local", "feature"], as_index=False)["Arithmetic Mean"]
        .mean()
    )

    met_wide = met_daily.pivot_table(
        index=["site_id", "Date Local"],
        columns="feature",
        values="Arithmetic Mean",
        aggfunc="mean",
    ).reset_index()

    analysis_df = no2_daily.merge(met_wide, on=["site_id", "Date Local"], how="inner")
    analysis_cols = ["NO2", "Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity"]
    analysis_df = analysis_df.dropna(subset=analysis_cols)

    print(f"Merged rows for analysis: {len(analysis_df):,}")
    print(f"Unique sites: {analysis_df['site_id'].nunique()}")
    display(analysis_df[analysis_cols].describe().T)

Meteorological source CSV files were not found in 'no2 analysis'.
Skipping met-feature analysis and regression sections until these files are added:
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_42602_2023.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_42602_2024.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_WIND_2023.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_WIND_2024.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_TEMP_2023.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_TEMP_2024.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_PRESS_2023.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_PRESS_2024.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_RH_DP_2023.csv
  - /mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/daily_RH_DP_2024.csv


In [5]:
if analysis_df.empty:
    print("Skipping correlation/scatter plots because meteorological source data is unavailable.")
else:
    # Pearson correlation matrix
    corr = analysis_df[["NO2", "Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity"]].corr(method="pearson")

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
    plt.title("Pearson Correlation Matrix: NO2 and Meteorological Features")
    plt.tight_layout()
    plt.show()

    display(corr)

    # Pairwise scatter plots: NO2 vs meteorological features
    pair_df = analysis_df[["NO2", "Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity"]].copy()

    g = sns.pairplot(
        pair_df,
        x_vars=["Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity"],
        y_vars=["NO2"],
        kind="scatter",
        height=3.0,
        aspect=1.1,
        plot_kws={"alpha": 0.35, "s": 14, "edgecolor": "none"},
    )

    g.fig.suptitle("NO2 vs Meteorological Features", y=1.03)
    plt.show()

Skipping correlation/scatter plots because meteorological source data is unavailable.


## Incremental scikit-learn Regression Pipeline

This section fits linear regression models incrementally:

1. Single predictor: Wind Speed
2. Two predictors: Wind Speed + Relative Humidity
3. Full model: Temperature, Wind Speed, Surface Pressure, Relative Humidity + harmonic Day-of-Year terms

For each model, we report:

- Learned coefficients
- MAE
- RMSE
- R2

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if analysis_df.empty:
    print("Skipping incremental regression pipeline because meteorological source data is unavailable.")
    reg_df = pd.DataFrame()
    model_specs = {}
    metrics_df = pd.DataFrame()
    coef_tables = {}
else:
    # Build regression dataframe from existing meteorology merge output.
    reg_df = analysis_df.copy()
    reg_df = reg_df.dropna(subset=["NO2", "Temperature", "Wind Speed", "Surface Pressure", "Relative Humidity", "Date Local"])

    # Harmonic DOY terms for the full model.
    doy = reg_df["Date Local"].dt.dayofyear + reg_df["Date Local"].dt.hour.fillna(0) / 24.0
    omega = 2.0 * np.pi / 365.25
    reg_df["doy_cos"] = np.cos(omega * doy)
    reg_df["doy_sin"] = np.sin(omega * doy)

    y = reg_df["NO2"].values

    model_specs = {
        "Single: Wind Speed": ["Wind Speed"],
        "Two: Wind Speed + Relative Humidity": ["Wind Speed", "Relative Humidity"],
        "Full: Weather + Harmonic DOY": [
            "Temperature",
            "Wind Speed",
            "Surface Pressure",
            "Relative Humidity",
            "doy_cos",
            "doy_sin",
        ],
    }

    rows = []
    coef_tables = {}

    for model_name, features in model_specs.items():
        X = reg_df[features].values

        pipe = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("regressor", LinearRegression()),
            ]
        )
        pipe.fit(X, y)
        y_pred = pipe.predict(X)

        mae = mean_absolute_error(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        r2 = r2_score(y, y_pred)

        coefs = pipe.named_steps["regressor"].coef_
        intercept = pipe.named_steps["regressor"].intercept_

        coef_df = pd.DataFrame({
            "feature": features,
            "coefficient": coefs,
        })
        coef_tables[model_name] = coef_df

        rows.append(
            {
                "Model": model_name,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "Intercept": intercept,
            }
        )

    metrics_df = pd.DataFrame(rows).set_index("Model")
    metrics_df = metrics_df[["MAE", "RMSE", "R2", "Intercept"]].round(4)

    print("Incremental Regression Metrics")
    display(metrics_df)

    print("\nLearned Coefficients")
    for model_name, coef_df in coef_tables.items():
        print(f"\n{model_name}")
        display(coef_df.round(4))

Skipping incremental regression pipeline because meteorological source data is unavailable.


## 3-Panel Diagnostics for Best Regression Model

This section identifies the best regression model from the incremental pipeline (lowest RMSE), then creates:

1. Actual vs Predicted NO2 over time
2. Actual vs Predicted scatter with 1:1 reference line
3. Residuals (Predicted - Actual) over time with horizontal lines at 0 and ±1 standard deviation, plus a residual histogram

In [7]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

if not model_specs or reg_df.empty:
    print("Skipping diagnostics figure because regression models were not trained (missing meteorological source data).")
else:
    # Refit models and store predictions so diagnostics are deterministic.
    model_predictions = {}
    model_metrics = []

    for model_name, features in model_specs.items():
        X = reg_df[features].values
        y_true = reg_df["NO2"].values

        pipe = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("regressor", LinearRegression()),
            ]
        )
        pipe.fit(X, y_true)
        y_pred = pipe.predict(X)

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        model_predictions[model_name] = {
            "y_true": y_true,
            "y_pred": y_pred,
            "time": reg_df["Date Local"].values,
        }
        model_metrics.append({"Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2})

    metrics_ranked = pd.DataFrame(model_metrics).sort_values("RMSE", ascending=True).reset_index(drop=True)
    best_model_name = metrics_ranked.loc[0, "Model"]
    best = model_predictions[best_model_name]

    # For clearer time-series visualization, aggregate to daily means.
    diag_df = pd.DataFrame(
        {
            "Date Local": pd.to_datetime(best["time"]),
            "Actual": best["y_true"],
            "Predicted": best["y_pred"],
        }
    ).sort_values("Date Local")

    diag_daily = (
        diag_df.set_index("Date Local")
        .resample("D")[["Actual", "Predicted"]]
        .mean()
        .dropna()
    )

    residual = diag_daily["Predicted"] - diag_daily["Actual"]
    resid_std = residual.std()

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.suptitle(f"Diagnostics - Best Regression Model: {best_model_name}", fontsize=14, y=1.02)

    # Panel 1: Time-series Actual vs Predicted
    axes[0].plot(diag_daily.index, diag_daily["Actual"], label="Actual", color="black", linewidth=1.6)
    axes[0].plot(diag_daily.index, diag_daily["Predicted"], label="Predicted", color="tab:blue", linewidth=1.4, alpha=0.9)
    axes[0].set_title("1) Actual vs Predicted NO2 Over Time")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("NO2")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Panel 2: Scatter with 1:1 line
    x = diag_daily["Actual"].values
    y = diag_daily["Predicted"].values
    axes[1].scatter(x, y, alpha=0.45, s=18, color="tab:green", edgecolor="none")
    lims_min = min(np.nanmin(x), np.nanmin(y))
    lims_max = max(np.nanmax(x), np.nanmax(y))
    axes[1].plot([lims_min, lims_max], [lims_min, lims_max], "r--", linewidth=1.5, label="1:1 line")
    axes[1].set_xlim(lims_min, lims_max)
    axes[1].set_ylim(lims_min, lims_max)
    axes[1].set_title("2) Actual vs Predicted Scatter")
    axes[1].set_xlabel("Actual NO2")
    axes[1].set_ylabel("Predicted NO2")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # Panel 3: Residuals over time + histogram inset
    axes[2].plot(diag_daily.index, residual.values, color="tab:purple", linewidth=1.3)
    axes[2].axhline(0.0, color="black", linestyle="-", linewidth=1.2, label="0")
    axes[2].axhline(+resid_std, color="tab:red", linestyle="--", linewidth=1.1, label="+1 std")
    axes[2].axhline(-resid_std, color="tab:red", linestyle="--", linewidth=1.1, label="-1 std")
    axes[2].set_title("3) Residuals Over Time (Predicted - Actual)")
    axes[2].set_xlabel("Date")
    axes[2].set_ylabel("Residual")
    axes[2].legend(loc="upper left")
    axes[2].grid(alpha=0.3)

    hist_ax = inset_axes(axes[2], width="36%", height="55%", loc="upper right", borderpad=1.0)
    hist_ax.hist(residual.values, bins=24, color="gray", alpha=0.8, edgecolor="white")
    hist_ax.axvline(0.0, color="black", linewidth=1.0)
    hist_ax.set_title("Residual Histogram", fontsize=9)
    hist_ax.tick_params(axis="both", labelsize=8)

    plt.tight_layout()
    plt.show()

    print("Best model metrics:")
    display(metrics_ranked.head(1).round(4))

Skipping diagnostics figure because regression models were not trained (missing meteorological source data).
